Subtopic 7: The Production Trifecta (Checkpointers + Retries + Interrupts)
When building an enterprise-grade AI system, you don't use these three architectural primitives in isolation. They form a unified safety and recovery lifecycle that guards your application against code exceptions, network instability, and unapproved AI actions.

Here is how the three features cooperate in harmony inside a single execution thread:

The Guard (Retry Policy): When a node encounters a flaky external API or database timeout, the RetryPolicy instantly handles the micro-recovery locally, preventing the whole graph from crashing.

The Vault (Checkpointer): Every time a node succeeds, the state is committed to disk. If a macro-failure occurs (like a total server power outage), the checkpointer ensures you don't lose progress.

The Brake (Interrupt Gate): For high-risk operations, the graph utilizes the database checkpoint to safely serialize the state, freeze the thread execution, and wait indefinitely for a human administrator to signal a resume.

In [10]:
from typing import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph, START, END
from langgraph.types import RetryPolicy
from langgraph.checkpoint.memory import MemorySaver

# ==========================================
# 1. THE STATE
# ==========================================
class OrderState(TypedDict):
    item_id: str
    price: float
    inventory_verified: bool
    billing_status: str
    logs: Annotated[list, operator.add]

# Global tracker to simulate a flaky network connection for our retry demonstration
API_ATTEMPTS = 0

In [11]:
# ==========================================
# 2. THE NODES + RETRY POLICY DEFINITION
# ==========================================

# We want this node to automatically retry if the network flakes out
flaky_network_policy = RetryPolicy(
    max_attempts=3,
    initial_interval=0.5,
    backoff_factor=2.0,
    retry_on=ConnectionError # Only retry on network issues, not code bugs
)

def verify_inventory_node(state: OrderState):
    global API_ATTEMPTS
    API_ATTEMPTS += 1
    
    print(f"📦 [Inventory Node] Attempt #{API_ATTEMPTS}: Checking stock for {state['item_id']}...")
    
    # Simulate a flaky API: Fail on the first attempt, succeed on the second!
    if API_ATTEMPTS < 2:
        print("⚠️ [Inventory Node] Network timeout! Triggering Retry Policy...")
        raise ConnectionError("Warehouse API did not respond in time.")
        
    print("✅ [Inventory Node] Stock confirmed. Writing checkpoint to database.")
    return {
        "inventory_verified": True,
        "logs": ["Inventory verified on attempt #2"]
    }


def charge_billing_node(state: OrderState):
    print(f"💳 [Billing Node] CRITICAL: Charging ${state['price']} to user's card...")
    return {
        "billing_status": "charged",
        "logs": ["Credit card successfully billed"]
    }


In [3]:
# ==========================================
# 3. WIRING THE GRAPH WITH AN INTERRUPT GATE
# ==========================================
builder = StateGraph(OrderState)

# Register nodes (Attaching the Retry Policy to our flaky inventory checker)
builder.add_node("inventory", verify_inventory_node, retry_policy=flaky_network_policy)
builder.add_node("billing", charge_billing_node)

# Set up roads
builder.add_edge(START, "inventory")
builder.add_edge("inventory", "billing") # The road layout is direct
builder.add_edge("billing", END)

# Initialize the checkpointer database
production_memory = MemorySaver()

# Compile the graph combining the Checkpointer AND the Interrupt Gate before billing
app = builder.compile(
    checkpointer=production_memory,
    interrupt_before=["billing"] # Safety brake!
)
print("🚀 Production Trifecta Graph successfully compiled!")

🚀 Production Trifecta Graph successfully compiled!


In [5]:
# ==========================================
# 4. EXECUTION PHASE 1: Triggering Retries and Pausing
# ==========================================
print("\n=== PHASE 1: Processing User Order ===")
config = {"configurable": {"thread_id": "order_abhishek_777"}}
initial_order = {
    "item_id": "ultra_gpu_2026",
    "price": 1499.99,
    "inventory_verified": False,
    "billing_status": "unbilled",
    "logs": ["Order submitted by user"]
}

# Run the graph. It will handle the retry automatically, then hit the interrupt gate.
app.invoke(initial_order, config=config)

# Inspect the database state to confirm it's safely paused at the checkpoint
snapshot = app.get_state(config)
print(f"\nIs graph paused? {len(snapshot.next) > 0}")
print(f"Next Node to run: {snapshot.next}")
print(f"Current State Logs: {snapshot.values['logs']}")



=== PHASE 1: Processing User Order ===
📦 [Inventory Node] Attempt #3: Checking stock for ultra_gpu_2026...
✅ [Inventory Node] Stock confirmed. Writing checkpoint to database.

Is graph paused? True
Next Node to run: ('billing',)
Current State Logs: ['Order submitted by user', 'Inventory verified on attempt #2', 'Credit card successfully billed', 'Order submitted by user', 'Inventory verified on attempt #2']


In [7]:

# ==========================================
# 5. EXECUTION PHASE 2: Human Sign-off & Resume
# ==========================================
print("\n=== PHASE 2: Admin Clicks 'Approve Payment' ===")

# Pass None to resume the frozen transaction thread out of the checkpointer
final_state = app.invoke(None, config=config)

print("\n--- FINAL SYSTEM STATE ---")
print(f"Billing Status: {final_state['billing_status']}")
print(f"Complete Audit Trail: {final_state['logs']}")


=== PHASE 2: Admin Clicks 'Approve Payment' ===

--- FINAL SYSTEM STATE ---
Billing Status: charged
Complete Audit Trail: ['Order submitted by user', 'Inventory verified on attempt #2', 'Credit card successfully billed', 'Order submitted by user', 'Inventory verified on attempt #2', 'Credit card successfully billed']
